In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.max_columns=1000

**Merge files**

In [2]:
train_n_batches = 40
test_n_batches = 10


df_train = pd.concat((
    pd.read_parquet(f"02_features/batch_{i}.parquet")
    for i in range(1, train_n_batches + 1)
))

df_test = pd.concat((
    pd.read_parquet(f"02_features/batch_{i}.parquet")
    for i in range(train_n_batches + 1, train_n_batches + test_n_batches + 1)
))

In [3]:
# df_train = df_train[ df_train["MaxMoveNumber"] >= 5 ]
# df_test = df_test[ df_test["MaxMoveNumber"] >= 5 ]

In [4]:
len(df_train), len(df_test)

(400000, 100000)

In [5]:
df_train.head()

,GameId,WhiteElo,BlackElo,Elo,Opening,ECO,FirstMoves,MeanStartLoss,MeanQueenLoss,MeanRookLoss,MeanKnightLoss,MeanBishopLoss,MeanKingLoss1,MeanKingLoss2,MeanPawnLoss1,MeanPawnLoss2,MeanFlagPawnLoss1,MeanFlagPawnLoss2,MeanCenterPawnLoss1,MeanCenterPawnLoss2
0,jgeqUmTV,1844,1819,1831,Scotch Game: Classical Variation,C45,e4 e5 Nf3 Nc6 d4 exd4 Nxd4 Bc5 Be3 d6 Nxc6 bxc...,0.000435,0.000000,0.000247,0.000251,0.000306,0.00469,0.000265,0.000641,0.000614,0.000643,0.000643,0.000596,0.000578
1,gdHoCJB5,1517,1504,1510,"Scotch Game: Classical Variation, Intermezzo V...",C45,e4 e5 Nf3 Nc6 d4 exd4 Nxd4 Bc5 Nxc6 Qf6 Be3 Bx...,0.000578,0.000684,0.000527,0.000449,0.000478,0.00469,0.000750,0.000825,0.000825,0.000000,0.000000,0.000929,0.000929
2,oVAuIb3u,2092,2096,2094,Indian Defense,A45,d4 Nf6 Bf4 g6 Nc3 Bg7 e4 d6 Qd2 c6 Bh6 Bxh6 Qx...,0.000592,0.000404,0.000761,0.001424,0.000679,0.01996,0.002040,0.000260,0.000281,0.000535,0.000383,0.000194,0.000226
3,cPMRdRO3,1684,1716,1700,Italian Game: Paris Defense,C50,e4 e5 Nf3 d6 Bc4 Nc6 Nc3 Be7 d4 f5 dxe5 Nxe5 N...,0.000368,0.000991,0.000430,0.000110,0.000302,0.01205,0.000608,0.000482,0.065285,0.000910,0.114672,0.000464,0.000464
4,dWmz6C3J,1653,1653,1653,Bishop's Opening: Berlin Defense,C24,e4 e5 Bc4 Nf6 d3 d5 Bb3 Bc5 Nc3 c6 Bg5 d4 Nce2...,0.000912,0.000079,0.000449,0.001499,0.000974,0.00469,0.001857,0.000377,0.000377,0.000221,0.000221,0.000480,0.000480


**Check features**

In [ ]:
feature = (df_train["MaxMoveNumber"] // 2).clip(0, 40)

In [ ]:
fig = px.line(
    feature.value_counts().sort_index()
)

fig.data[0].mode = "lines+markers"
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

In [ ]:
fig = px.line(
    df_train.groupby(feature).agg({"Elo": "mean"})
)

fig.data[0].mode = "lines+markers"
fig.update_layout(
    template="plotly_white",
    showlegend=False
)

fig.show()

**Saving files**

In [6]:
def bin_features(df):
    
    df_new = pd.DataFrame()

    for feature in df.columns:
        if feature in ["GameId", "WhiteElo", "BlackElo", "Elo", "Opening", "ECO", "FirstMoves"]:
            df_new[feature] = df[feature]
        else:
            df_new[feature] = df[feature].clip(
                df[feature].quantile(0.005), 
                df[feature].quantile(0.995)
            )

    # # Ошибки
    # df["NBlunders"] = (df["NBlunders"]).clip(0, 20)
    # df["NOkayMoves"] = (df["NOkayMoves"] // 5).clip(0, 30)
    # df["MeanBlunders"] = (df["MeanBlunders"] // 0.02).clip(0, 15)
    # df["MeanMistakes"] = (df["MeanMistakes"] // 0.05).clip(0, 10)
    # df["MeanBadMoves"] = (df["MeanBadMoves"] // 0.02).clip(0, 24)
    # df["MeanOkayMoves"] = (df["MeanOkayMoves"] // 0.02).clip(20, 50)
    
    # # Средний ход ошибок
    # df["MoveNumberBlunder"] = (df["MoveNumberBlunder"] // 2).clip(0, 25)
    # df["MoveNumberMistake"] = (df["MoveNumberMistake"] // 3).clip(0, 15)
    # df["MoveNumberBadMove"] = (df["MoveNumberBadMove"] // 3).clip(0, 15)
    
    # # Eval
    # df["MeanAbsEval"] = (df["MeanAbsEval"] // 20).clip(0, 40)
    # df["EvalStd"] = (df["EvalStd"] // 50).clip(0, 18)
    # df["NEqualGame300"] = (df["NEqualGame300"] // 3).clip(0, 30)
    # df["MeanLostGame600"] = (df["MeanLostGame600"] // 0.05).clip(0, 18)
    
    # df["AbsEvalMedian"] = (df["AbsEvalMedian"] // 10)
    # df["CentipawnLossMedian"] = (df["CentipawnLossMedian"] // 10).clip(0, 20)
    
    # # Потери сантипешек
    # df["MeanCentipawnLoss"] = (df["MeanCentipawnLoss"] // 10).clip(0, 22)
    # df["StartCentipawnLoss15"] = (df["StartCentipawnLoss15"] // 10).clip(0, 25)
    # df["KnightCentipawnLoss"] = (df["KnightCentipawnLoss"] // 20).clip(0, 16)
    # df["PawnCentipawnLoss"] = (df["PawnCentipawnLoss"] // 10).clip(1, 20)
    
    # # Прочее
    # df["MeanHasMate"] = (df["MeanHasMate"] // 0.05).clip(0, 8)
    # df["MeanChecks"] = (df["MeanChecks"] // 0.02).clip(0, 12)
    # df["NMoves"] = (df["NMoves"] // 5).clip(0, 20)
    
    # # WinOdds
    # df["WinOddsStd"] = (df["WinOddsStd"] // 0.01).clip(2, 60)
    # df["WinOddsMean"] = (df["WinOddsMean"].abs() // 0.01).clip(0, 70)
    # df["MaxAdvLost"] = (df["MaxAdvLost"] // 0.05).clip(2, 30)
    # df["MeanAdvLost"] = (df["MeanAdvLost"] // 0.01).clip(2, 30)
    # df["StartAdvLost10"] = (df["StartAdvLost10"] // 0.01).clip(0, 30)
    
    # df["WinOddsMedian"] = (df["WinOddsMedian"].abs() // 0.01)
    # df["MedianAdvLost"] = (df["MedianAdvLost"] // 0.005).clip(0, 20)
    
    # # df["TimeSpentMean"] = (df["TimeSpentMean"] // 1).clip(-2, 25)
    
    
    return df_new

In [7]:
bin_features(df_train).to_parquet("03_datasets/binned_train.parquet")
bin_features(df_test).to_parquet("03_datasets/binned_test.parquet")